# 06 — Direct Preference Optimization (DPO) Training
**Goal**: Train the SFT model using Direct Preference Optimization (DPO) on execution debug trajectory pairs ($ \beta=0.1 $, LR $5\times 10^{-5}$). Produces `./checkpoints/dpo/final` for RQ5 comparison against PPO.

---

## Step 1: Environment Setup & Universal Path Resolution

In [ ]:
import sys, os

# Universal Path Resolution for Kaggle / Colab / Molab / Local
def find_repo_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.isdir(os.path.join(curr, 'src')):
            return curr
        curr = os.path.dirname(curr)
    for fallback in ['/kaggle/working/self-correction-llm-rl', '/kaggle/working', '/content/self-correction-llm-rl', '/content']:
        if os.path.isdir(os.path.join(fallback, 'src')):
            return fallback
    return os.path.abspath('..')

repo_root = find_repo_root()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project root added to sys.path: {repo_root}")

import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.training.dpo import make_preference_pairs, run_dpo_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

## Step 2: Collect Preference Trajectory Pairs `(prompt, chosen, rejected)`
Runs $K=3$ debug rollouts on APPS problems. Extracts winning solutions (AC) as `chosen` and failing solutions (CE/RE/WA) as `rejected`.

In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"

model_path = SFT_CHECKPOINT if os.path.exists(SFT_CHECKPOINT) else MODEL_NAME
print(f"Loading model from {model_path} for DPO pair collection...")
model, tokenizer = load_model_and_tokenizer(model_name=model_path, load_in_4bit=True)

print("\nLoading APPS dataset for preference pair generation...")
apps = load_dataset('codeparrot/apps', split='train[:500]', trust_remote_code=True)
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)

print("Generating preference dataset...")
preference_data = make_preference_pairs(apps_clean, model, tokenizer, K=3)
print(f"Total DPO preference pairs generated: {len(preference_data)}")

## Step 3: Run DPO Training
Uses PEFT reference model trick to avoid loading duplicate reference weights in VRAM.

In [ ]:
print("Starting DPO Training...")
dpo_trainer = run_dpo_training(
    model=model,
    tokenizer=tokenizer,
    preference_data=preference_data,
    output_dir="./checkpoints/dpo",
    beta=0.1,
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
)

print("\nDPO Training completed successfully!")
print("Saved final DPO adapter checkpoint to ./checkpoints/dpo/final")